In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt

from typing import *
from pathlib import Path

ROOT_DIR = Path(os.getcwd()).parent

%matplotlib inline

In [24]:
df = pd.read_csv(os.path.join(ROOT_DIR, "data/pumps/labeled_chats.csv"))
df["timestamp"] = pd.to_datetime(df["timestamp"])
df.head(2)

,message,timestamp,coin,exchange,is_release,pump_id,is_anomaly,chat_id
0,*ANNONCEMENT*\n\nGreetings to all members of t...,2021-11-18 10:34:27+00:00,MODEFI-USDT,Kucoin,0.0,198.0,NaN,cryptoclubpump
1,"Hi Team,\n\nI’m still hoping we will find a pr...",2021-11-18 12:42:31+00:00,MODEFI-USDT,Kucoin,0.0,198.0,NaN,cryptoclubpump


In [25]:
df = df[df["is_release"] == 1].copy()
df["time_without_seconds"] = df["timestamp"].dt.floor("min")

unique_pumps = []

for _, df_group in df.groupby("time_without_seconds"):
    df_pump = (
        df_group[df_group["timestamp"] == df_group["timestamp"].min()][["coin", "timestamp", "exchange"]]
    )
    coin, time, exchange = df_pump.iloc[0, :]

    unique_pumps.append({
        "ticker": coin,
        "time": time,
        "exchange": exchange
    })

In [26]:
df_pumps = pd.DataFrame(unique_pumps)
df_pumps.to_csv("pumps.csv", index=False)

<h4>Collect Fantazzini pumps from his paper</h4>

In [7]:
from selectolax.parser import HTMLParser
import requests

In [8]:
resp = requests.get(
    "https://www.mdpi.com/2225-1146/11/3/22"
)

dom = HTMLParser(resp.text)

rows = dom.css("#table_body_display_econometrics-11-00022-t002 tbody > tr")
data = []

for row in rows:
    values = [el.text() for el in row.css("td")]
    tickers = values[0::2]
    times = values[1::2]

    for ticker, time in zip(tickers, times):
        data.append({
            "ticker": ticker+"BTC",
            "time": time
        })

df_fan = pd.DataFrame(data)
df_fan = df_fan.iloc[:-1]
df_fan["time"] = pd.to_datetime(df_fan["time"])

df_fan["exchange"] = "Binance"
df_fan.head()

,ticker,time,exchange
0,ADXBTC,2022-09-20 15:56:00,Binance
1,ATMBTC,2022-01-12 17:37:00,Binance
2,SNTBTC,2021-10-23 16:31:00,Binance
3,NASBTC,2021-08-22 17:00:00,Binance
4,IRISBTC,2022-09-17 13:24:00,Binance


In [9]:
df_fan.to_csv(os.path.join(ROOT_DIR, "data/pumps/fantazzini_pumps.csv"), index=False)

<p>Here we will check Fantazzini pumps data, we will load second klines from Binance and check if there was a spike big enough to label it as a pump</p>

In [4]:
cols = [
    "open_time", "open", "high", "low", "close",
    "volume", "close_time", "quote_vol", "num_trades",
    "taker_buy_base_vol", "taker_buy_quote_vol", "ignore"
]

In [5]:
from urllib.parse import urljoin

import zipfile
import io

def query_data(ticker: str, pump_date: pd.Timestamp) -> pd.DataFrame:

    response = requests.get(
        urljoin(
            "https://data.binance.vision/data/spot/daily/klines/",
            f"{ticker}/1s/{ticker}-1s-{pump_date.year}-{str(pump_date.month).zfill(2)}-{str(pump_date.day).zfill(2)}.zip"
        )
    )

    with zipfile.ZipFile(io.BytesIO(response.content), "r") as zip_ref:
        for file_name in zip_ref.namelist():
            file_content = io.StringIO(zip_ref.read(file_name).decode("utf-8"))

    df = pd.read_csv(file_content, header=None, names=cols)

    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")

    df = df.convert_dtypes()

    return df

In [8]:
ticker = "NASBTC"
pump_date = pd.Timestamp('2021-08-22 17:00:00')

df_pump = query_data(ticker=ticker, pump_date=pump_date)
df_pump.head(2)

,open_time,open,high,low,close,volume,close_time,quote_vol,num_trades,taker_buy_base_vol,taker_buy_quote_vol,ignore
0,2021-08-22 00:00:00,0.000013,0.000013,0.000013,0.000013,0.0,2021-08-22 00:00:00.999,0.0,0,0.0,0.0,0
1,2021-08-22 00:00:01,0.000013,0.000013,0.000013,0.000013,0.0,2021-08-22 00:00:01.999,0.0,0,0.0,0.0,0


In [9]:
import plotly.graph_objects as go

In [14]:
df_pump = df_pump[
    (df_pump["open_time"] >= pump_date - pd.Timedelta(hours=1)) &
    (df_pump["open_time"] <= pump_date + pd.Timedelta(hours=1))
].copy()

candlestick_trace = go.Candlestick(
    x=df_pump["open_time"],
    open=df_pump["open"],
    high=df_pump["high"],
    low=df_pump["low"],
    close=df_pump["close"],
    name="Price"
)
# Create the figure
fig = go.Figure(data=[candlestick_trace])

# Display the graph
fig.show()
# plt.axvline(x=pump_date, color="red", linestyle="--", alpha=.4)

In [47]:
from tqdm import tqdm
from IPython import display


class ManualPumpCheck:

    def query_data(self, ticker: str, pump_date: pd.Timestamp) -> pd.DataFrame:
        
        response = requests.get(
            urljoin(
                "https://data.binance.vision/data/spot/daily/klines/",
                f"{ticker}/1s/{ticker}-1s-{pump_date.year}-{str(pump_date.month).zfill(2)}-{str(pump_date.day).zfill(2)}.zip"
            )
        )

        with zipfile.ZipFile(io.BytesIO(response.content), "r") as zip_ref:
            for file_name in zip_ref.namelist():
                file_content = io.StringIO(zip_ref.read(file_name).decode("utf-8"))

        df = pd.read_csv(file_content, header=None, names=cols)

        df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
        df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")

        df = df.convert_dtypes()

        return df
    
    def check_pumps(self, df: pd.DataFrame) -> pd.DataFrame:
        filtered_pumps: List[Dict[str, Any]] = []

        for pump in tqdm(df.itertuples()):
            try:
                data: pd.DataFrame = self.query_data(ticker=pump.ticker, pump_date=pump.time)
            except:
                print(f"Problem with ticker {pump.ticker}")
                continue

            plt.clf()

            data.plot(x="open_time", y="open")
            plt.axvline(x=pump.time, color="red", linestyle="--", alpha=.45)
            plt.show()

            while True:
                is_pump = input("Is this a pump - yes(1), no(0): ")
                if is_pump in ["0", "1"]:
                    break
            
            if int(is_pump):
                filtered_pumps.append({
                    "ticker": pump.ticker,
                    "time": pump.time
                })

            display.clear_output(wait=True)

        return pd.DataFrame(filtered_pumps)

In [21]:
df_fan_clean = ManualPumpCheck().check_pumps(df_fan)
df_fan_clean.head()

,ticker,time
0,ADXBTC,2022-09-20 15:56:00
1,NASBTC,2021-08-22 17:00:00
2,POABTC,2021-10-21 16:00:00
3,NEBLBTC,2022-01-02 17:00:00
4,WRXBTC,2021-08-20 11:40:00


In [19]:
df_fan_clean.to_csv(os.path.join(ROOT_DIR, "data/pumps/fantazzini_verified_pumps.csv"), index=False)

<p>Checking pumps collected from telegram chats</p>

In [34]:
df_binance = df_pumps[df_pumps["exchange"] == "Binance"].copy()
df_binance["ticker"] = df_binance["ticker"].str.replace("-", "")
df_binance.head(2)

,ticker,time,exchange
0,STORJBTC,2018-12-08 18:45:00+00:00,Binance
1,WABIBTC,2018-12-16 17:00:04+00:00,Binance


In [49]:
df_binance_clean = ManualPumpCheck().check_pumps(df_binance)
df_binance_clean.head()

89it [09:41,  6.53s/it]


,ticker,time
0,STORJBTC,2018-12-08 18:45:00+00:00
1,WABIBTC,2018-12-16 17:00:04+00:00
2,NEBLBTC,2018-12-23 16:59:55+00:00
3,GXSBTC,2019-01-20 17:59:19+00:00
4,HCBTC,2019-01-25 17:59:58+00:00


In [53]:
df_binance_clean.to_csv(os.path.join(ROOT_DIR, "data/pumps/telegram_pumps_clean.csv"), index=False)

In [59]:
df_binance_clean["source"] = "telegram"
df_fan_clean["source"] = "Fantazzini paper"

df_clean_pumps = pd.concat([df_binance_clean, df_fan_clean], axis=0)
df_clean_pumps = df_clean_pumps.reset_index(drop=True)

df_clean_pumps.to_csv(
    os.path.join(ROOT_DIR, "data/pumps/cleaned/pumps_verified.csv"), index=False
)